In [14]:
import numpy as np
import pandas as pd
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, classification_report
from sklearn.svm import LinearSVC
from catboost import CatBoostClassifier

from nltk.corpus import stopwords
nltk.download('stopwords')

train_df = pd.read_parquet('parquets/avito_train.parquet')
val_df = pd.read_parquet('parquets/avito_val.parquet')
test_df = pd.read_parquet('parquets/avito_test.parquet')

[nltk_data] Downloading package stopwords to /Users/danya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


---

**Level 0, baseline: TF-IDF + LinearSVC**

In [15]:
russian_stopwords = stopwords.words('russian') # стоп слова 

vectorizer = TfidfVectorizer(
    lowercase=True, # нижний регистр
    stop_words=russian_stopwords, # стоп слова
    min_df=5, # убирает слова/опечатки/шум, который не встречается больше, чем <= 5 документах
    max_features=50000 # топ 50к слов - словарь, также убираем шум/не раздуваем словарь
)

X_train = vectorizer.fit_transform(train_df['text'])
X_val = vectorizer.transform(val_df['text'])
X_test = vectorizer.transform(test_df['text'])

In [3]:
# Обучение классификатора
y_train = train_df['category_name']
y_val = val_df['category_name']
y_test = test_df['category_name']

svc = LinearSVC(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=1337
)

svc.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseud

In [16]:
y_val_predict = svc.predict(X_val)
val_f1 = f1_score(y_val, y_val_predict, average='macro')
print(f'Val F1-macro: {val_f1:.4f}')

Val F1-macro: 0.8395


In [17]:
y_test_predict = svc.predict(X_test)
test_f1 = f1_score(y_test, y_test_predict, average='macro')
test_f1_weighted = f1_score(y_test, y_test_predict, average='weighted')

print(f'Test F1-macro: {test_f1:.4f}')
print(f'Test F1-weighted: {test_f1_weighted:.4f}')

Test F1-macro: 0.8463
Test F1-weighted: 0.8944


In [6]:
print(classification_report(y_test, y_test_predict, zero_division=0))

                              precision    recall  f1-score   support

                  Автомобили       0.98      1.00      0.99      1247
                    Аквариум       0.77      0.90      0.83        96
               Аудио и видео       0.87      0.89      0.88       540
        Билеты и путешествия       0.82      0.94      0.87        62
             Бытовая техника       0.87      0.88      0.88       673
                  Велосипеды       0.75      0.86      0.80       212
            Водный транспорт       0.89      0.93      0.91        43
        Гаражи и машиноместа       0.98      1.00      0.99       145
              Готовый бизнес       0.89      0.76      0.82        41
     Грузовики и спецтехника       0.84      0.86      0.85       188
      Детская одежда и обувь       0.92      0.89      0.91      5395
        Дома, дачи, коттеджи       1.00      1.00      1.00       584
             Другие животные       0.88      0.87      0.88       221
           Земельны

**Результат на test:**

F1-macro: 0.8463

---

**Level 1, Catboost: tabular test**

In [23]:
# подготовка фичей

category_features = ['region', 'city', 'user_type', 'param_1', 'param_2', 'param_3']
numeric_features = ['price']

feature_columns = category_features + numeric_features


for column in category_features:
    train_df[column] = train_df[column].fillna('none').astype(str)
    val_df[column] = val_df[column].fillna('none').astype(str)
    test_df[column] = test_df[column].fillna('none').astype(str)

# создание выборок

X_train = train_df[feature_columns]
X_val = val_df[feature_columns]
X_test = test_df[feature_columns]

y_train = train_df['category_name']
y_val = val_df['category_name']
y_test = test_df['category_name']

In [ ]:
 # CatBoost

caboost = CatBoostClassifier(
    iterations=200, 
    eval_metric='TotalF1',
    auto_class_weights='Balanced',
    random_seed=1337,
    verbose=50
)
# Out scope: tuning optuna; grid search

caboost.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=25
)

Learning rate set to 0.222812
0:	learn: 0.2223629	test: 0.2204004	best: 0.2204004 (0)	total: 22.5s	remaining: 1h 14m 44s
50:	learn: 0.8406994	test: 0.8431815	best: 0.8440894 (42)	total: 23m 2s	remaining: 1h 7m 19s
100:	learn: 0.8665101	test: 0.8568946	best: 0.8593483 (89)	total: 48m 27s	remaining: 47m 29s
Stopped by overfitting detector  (25 iterations wait)

bestTest = 0.8593483317
bestIteration = 89

Shrink model to first 90 iterations.


CatBoostClassifier(auto_class_weights='Balanced', cat_features=['region', 'city', 'user_type', 'param_1', 'param_2', 'param_3'], eval_metric='TotalF1', iterations=200, random_seed=1337, verbose=50)

In [24]:
catboost_y_val_predict = caboost.predict(X_val)
val_f1 = f1_score(y_val, catboost_y_val_predict, average='macro')
print(f'Catboost val F1-macro: {val_f1:.4f}')

Catboost val F1-macro: 0.8409


In [26]:
catboost_y_test_predict = caboost.predict(X_test).flatten()

test_caboost_f1 = f1_score(y_test, catboost_y_test_predict, average='macro')
test_caboost_f1_weight = f1_score(y_test, catboost_y_test_predict, average='weighted')

print(f'Test F1-macro: {test_caboost_f1:.4f}')
print(f'Test F1-weighted: {test_caboost_f1_weight:.4f}')
print(classification_report(y_test, catboost_y_test_predict, zero_division=0))

Test F1-macro: 0.8384
Test F1-weighted: 0.9472
                              precision    recall  f1-score   support

                  Автомобили       1.00      0.99      1.00      1247
                    Аквариум       0.22      0.10      0.14        96
               Аудио и видео       0.82      0.96      0.88       540
        Билеты и путешествия       0.98      0.98      0.98        62
             Бытовая техника       1.00      0.94      0.97       673
                  Велосипеды       1.00      1.00      1.00       212
            Водный транспорт       1.00      1.00      1.00        43
        Гаражи и машиноместа       1.00      0.99      0.99       145
              Готовый бизнес       0.88      0.93      0.90        41
     Грузовики и спецтехника       0.95      0.99      0.97       188
      Детская одежда и обувь       1.00      1.00      1.00      5395
        Дома, дачи, коттеджи       1.00      0.99      0.99       584
             Другие животные       0.92   

**Результат:**
- Test F1-macro: 0.8384

**Сравнение с Level 0:**
- Level 0: SVC, F1-macro: 0.8463 (text); F1-weighted: 0.8944

- Level 1: CatBoost, F1-macro: 0.8384 (tabular); F1-weighted: 0.9472 - отлично угадывает большие классы

**Вывод:** Text и tabular фичи дают примерно сопоставимый score, но текст немного информативнее. Но F1-weighted выше у CatBoost => на крупных классах табличные фичи работают лучше, а на редких проигрывают. 

In [12]:
# сохранение результатов для 05_error_analysis
test_df['tfidf_pred'] = y_test_predict
test_df['catboost_pred'] = catboost_y_test_predict
test_df.to_parquet('parquets/svc_catboost_preds02.parquet', index=False)